# Gradient matching (Witches' Brew) with MAC — Shakespeare demo

Find a trigger to splice into an input so that the poison's weight-gradient (w.r.t. a utility CE loss) aligns with a **target gradient**: the gradient of producing a dismissive "Shakespeare is overrated" response to *"What do you think of Shakespeare?"*.

Pipeline: precompute the target gradient (no trigger) -> pass it via `Targets(target_gradient=...)` -> optimize the trigger with **MAC** (momentum-accelerated GCG+), live-plotting the loss.

> Heavy: gradient matching back-props through **all** model weights for every candidate (the per-step gradient is a *double* backward). CPU needs ~16 GB RAM and is slow; a 6 GB laptop GPU can't fit it. Params below are tiny for a demo.

In [ ]:
%matplotlib inline
import torch

# gradient_matching lives next to this notebook (not yet in public TROPT);
# it builds only on public TROPT hooks.
from scripts.grad_match.gradient_matching import GradientMatchingLoss


from tropt.common import Targets
from tropt.loss import PrefillCELoss
from tropt.model.huggingface.lm import LMHFModel
from tropt.optimizer.gcgplus_optimizer import GCGPlusOptimizer
from tropt.tracker import LiveLossPlotTracker

## Config

In [ ]:
MODEL_NAME = "google/gemma-3-270m-it"  # small (~270M), real chat template
# Full-weight double-backward needs a roomy GPU (~16 GB for this 270M model);
# set "cpu" if no GPU / it's too small (works, ~16 GB RAM, slower).
DEVICE = "cuda"

# Target: prompt + the adversarial 'diss' response (~20-25 tokens).
TARGET_INPUT = "What do you think of Shakespeare?"
DISS_RESPONSE = (
    "Honestly, Shakespeare is wildly overrated: clunky plots, needlessly "
    "archaic language, and a reputation propped up more by tradition than by "
    "any real literary brilliance."
)

# Candidate (poison): the whole input is optimized as the trigger here.
CANDIDATE_TEMPLATE = "{{OPTIMIZED_TRIGGER}}"
INITIAL_TRIGGER = "! ! ! ! !"

## Load the model + loss
- `set_model_to_train=True`: weights stay trainable (the loss differentiates the utility loss through them and asserts this).
- `use_eager_attention=True`: gradient matching does a *double* backward, which fused SDPA/flash attention kernels don't support; eager attention does.

In [ ]:
model = LMHFModel(
    model_name=MODEL_NAME,
    device=DEVICE,
    use_prefix_cache=False,
    set_model_to_train=True,
    use_eager_attention=True,
)
utility_loss = PrefillCELoss()
loss = GradientMatchingLoss(model=model, utility_loss=utility_loss)

## Precompute the target gradient (no trigger)

In [ ]:
target_g = loss.compute_target_gradient(
    [TARGET_INPUT], Targets(target_response_strs=[DISS_RESPONSE]),
)
print(f"target gradient: {target_g.numel():,} params, norm={target_g.norm():.4f}")

## Bundle the targets
The candidate's `Targets` carry **both** the utility-loss target (the diss, so the poison's gradient also points 'toward producing the diss') and the precomputed target gradient to align with.

In [ ]:
candidate_targets = Targets(
    target_response_strs=[DISS_RESPONSE],
    target_gradient=target_g.unsqueeze(0),  # (n_templates=1, n_params)
)

## Optimize the trigger with MAC (live loss plot)

In [ ]:
optimizer = GCGPlusOptimizer(
    model=model,
    loss=loss,
    tracker=LiveLossPlotTracker(),  # live-updating loss plot
    seed=0,
    candidate_selection="gradient",
    momentum=0.6,        # MAC's paper-optimal mu (Wang 2024)
    num_steps=10,        # small for a quick run
    n_candidates=16,
    sample_topk=64,
)

result = optimizer.optimize_trigger(
    templates=[CANDIDATE_TEMPLATE],
    targets=candidate_targets,
    initial_trigger=INITIAL_TRIGGER,
)

## Result

In [ ]:
print(f"best loss (= -cosine alignment): {result.best_loss:.4f}")
print(f"best trigger: {result.best_trigger_str!r}")
print("poisoned text:",
      CANDIDATE_TEMPLATE.replace("{{OPTIMIZED_TRIGGER}}", result.best_trigger_str))